# Offline activity module IV

Unfortunately after 6-8 hours of endless trying to optimise and fit my dataset into the very restrictive dataframe expectations of TabPFN (specifically for time series dataframes), I did not succeed in using TabPFN to predict my data.\
Thus, I have decided against using my data and will just briefly discuss my experience with TabPFN and the output of the example dataset by huggingface they use in their standard script.\
The initial goal was to use my carbon source growth screen with OD600 data over time to predict growth curves with TabPFN. I had thought of predicting parts of existing growth curves but\
also predicting growth curves where there were issues with data collection and the datasets are incomplete to see whether TabPFN could infer the course of these datasets had they run normally.\
Please excuse my failure. My lack in background for computer sciences meant I had to read into many things (even pandas) for the first time and had to spend hours upon hours trying to get this to work\
but to no avail. I'll paste whatever code I have here for you to look into my best try but I cannot muster up more than this (this is only what I have left after countless of troubleshooting runs).

In [1]:
import os
import csv
import json
import numpy as np
import pandas as pd

In [40]:
"""
This is not a very special or interesting dataset. Since I haven't generated any more interesting data yet, I'm using some growth curve screens of V. cholerae on different
sole carbon sources like amino acids, sugars, organic acids, etc. The growth curves are recorded as OD600 data measured every 5 minutes for 72 hours. Most curves have the
characteristic sigmoid shape but there are some diauxic or otherwise special outliers. I also have data from an experiment where the plate reader was interrupted prematurely,
so the data cuts off after 8 hours. Perhaps TabPFN can predict the curve for some of the complete data or even a solid guess for the incomplete data? Let's write the data into
a json file with the correct shape, before it can be read in by pandas.
"""
with open('growthcurve_data_tabpfn.csv','r') as data:
    csvfile = list(csv.reader(data))

    # Extract the timepoint vector
    t = []
    for line in csvfile[1:]:
        tp = round(float(line[0]),3)
        t.append(tp)

    # Extract the desired data
    dat = {}
    for l in range(len(csvfile)):
        if l == 0:
            for val in range(len(csvfile[l])):
                if val != 0:
                    dat[csvfile[l][val]] = []
        else:
            for val in range(len(csvfile[l])):
                if val != 0:
                    if csvfile[l][val] == '':
                        continue
                    else:
                        value = float(csvfile[l][val])
                        dat[csvfile[0][val]].append(value)

# Write the data into a json file
with open("tabpfn_exercise.json", "w") as file:
    jsonfile = []
    for k, v in dat.items():
        jsonline = {'item_id' : k, 'timestamp' : t[:len(v)], 'target' : v}
        jsonfile.append(jsonline)
    json.dump(jsonfile,file)

In [38]:
# Read in the data as a pandas dataframe
df = pd.read_json('tabpfn_exercise.json')
df = df.reset_index()
ndf = pd.DataFrame(columns=['index', 'item_id', 'timestamp', 'target'])
rowindex = 0
for index, row in df.iterrows():
    for i in range(len(row['timestamp'])):
        ndf.loc[rowindex] = [index, row['item_id'], row['timestamp'][i], row['target'][i]]
        rowindex += 1
ndf["timestamp"] = pd.to_timedelta(ndf["timestamp"],unit='h')
ndf["timestamp"] = ndf["timestamp"].dt.round("5min")
ndf["timestamp"] = (pd.to_datetime('2026-05-14') + ndf["timestamp"])
ndf = ndf.sort_values(["item_id", "timestamp"])
ndf = ndf.drop_duplicates(["item_id", "timestamp"])
ndf.head()

# old unnecessary code
# for index, row in ndf.iterrows():
#     # ndf.at[index,'timestamp'] = pd.to_datetime('2026-05-14') + pd.to_timedelta(row['timestamp'],unit='h')
#     ndf.at[index,'timestamp'] = pd.to_timedelta(row['timestamp'],unit='h')
# for index, row in ndf.iterrows():
#     ndf.at[index, 'timestamp'] = row['timestamp'].dt.round('5min')

,index,item_id,timestamp,target
0,0,Acetic acid,0 days 00:00:00,0.111
1,0,Acetic acid,0 days 00:05:00,0.111
2,0,Acetic acid,0 days 00:10:00,0.111
3,0,Acetic acid,0 days 00:15:00,0.111
4,0,Acetic acid,0 days 00:20:00,0.111


In [ ]:
from datasets import load_dataset
from tabpfn_time_series import TimeSeriesDataFrame
from tabpfn_time_series.data_preparation import generate_test_X

prediction_length = 24
ndf["timestamp"] = pd.DatetimeIndex(ndf["timestamp"]).floor("h")
tsdf = TimeSeriesDataFrame.from_data_frame(ndf)
tsdf = tsdf[tsdf.index.get_level_values("item_id").isin(tsdf.item_ids[:10])]
train_tsdf, test_tsdf_ground_truth = tsdf.train_test_split(prediction_length=prediction_length)
test_tsdf = generate_test_X(train_tsdf, prediction_length)

In [ ]:
from tabpfn_time_series.plot import plot_actual_ts

plot_actual_ts(train_tsdf, test_tsdf_ground_truth)

In [ ]:
from tabpfn_time_series import TabPFNMode, TabPFNTimeSeriesPredictor

predictor = TabPFNTimeSeriesPredictor(
    tabpfn_mode=TabPFNMode.CLIENT,  # adapt this to TabPFNMode.CLIENT if using API
)

pred = predictor.predict(train_tsdf, test_tsdf) # This is how far I could get. Starting here the prediction was not possible despite all my best efforts. Perhaps you can spot the problem.

## TabPFN example datasets
Since it doesn't work for my data I'll evaluate the time series and classification applications of TabPFN. It is important to note that they might have picked these examples\
because their model works best with these, so I would always take it with a grain of salt.

TabPFN does seem to predict classifications for the Parkinson's dataset quite a bit more accurately than XGboost (TabPFN ROC AUC Score: 0.9777\
XGBoost ROC AUC Score: 0.9196). However, this is not to say that XGboost does a terrible job. Both models can reliably classify the patients.\
It's just that TabPFN is is correct 5% more often.\
 \
Now if we graphically compare the different prediction models to TabPFN it might on a first glance seem like TabPFN performs much better compared\
to the other models. But Prior Labs is trying to mislead us massively with this graph. The y-axis is manipulated to only show the top 5% (instead\
of 0-1 it is 0.95-1). This means that the actual improvement between TabPFN and other models is marginal. It still improves prediction but I wouldn't\
immediately trust this and call it a major improvement in prediction modelling.

![COMPARISON](model_comparison.png)

As a final step, I want to have a look at the time series prediction I wanted to use initially. I must say the prediction is surprisingly accurate here\
which is very good. I would have liked to see how TabPFN performs with my data but alas I have given up on trying more.

![TIME](timeseries_pred_example.png)

# Final verdict
I can see how TabPFN seems attractive and looks great as a predictive model overall. It seems to be relatively robust and predicts multiple\
different applications very well even outperforming all other state-of-the-art prediction models even if only marginally and not as monumentally\
as the creators might describe it.\
\
Nonetheless I can't help but be massively disappointed after trying to make it work for hours. As someone with hands-on lab experience and generally\
as a wet lab scientist TabPFN seems way too restrictive and sensitive to the irrationalities of real world data to be particularily useful. This might\
of course be limited to the time series prediction application but the dataset restrictions are frankly ridiculous in most cases and it just isn't\
realistic to adjust a real-world dataset for hours upon hours to receive a prediction for which you would still need actual experimental data to back it\
up. Obviously, this is likely just a general limitation of prediction models and computation but this does not change my disappointment.\
\
Hence, TabPFN does not appear particularly useful to me and my applications and I'm not fully convinced yet.